# CH101 Wonder3D multiview experiment

This notebook is a research-only extension of the CH101 AI pipeline. It uses the pinned Wonder3D repository to generate six consistent RGB/normal views from the approved front reference, extracts one mesh with NeuS, and sends it through the existing non-production evaluation gate. It never enables Unity input or Gate B.

The adaptive runner checks the Colab runtime before heavy setup. With a GPU it continues Wonder3D and NeuS; without one it completes the No-GPU validation workstream and stops before CUDA installation. The model, checkpoint terms, CUDA dependencies, and mesh extraction may still fail; every failure is recorded and no failure can promote a candidate.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

CHARACTER_CODE = 'CH101'
TOOLS_REPO_URL = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_REF = os.environ.get('RE_CAMP_BLENDER_TOOLS_REF', 'feature/ch101-free-ai3d-autobuild')
ART_REPO_URL = 'https://github.com/siri2677/re-camp.git'
ART_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
WONDER3D_REPO_URL = 'https://github.com/xxlong0/Wonder3D.git'
WONDER3D_COMMIT = 'd894f827aa8c2917761a0dad3ab40df74c7a5b24'
WONDER3D_INFERENCE_SCRIPT = 'test_mvdiffusion_seq.py'
WONDER3D_EXPECTED_VIEWS = 6
RUNTIME_NAME = os.environ.get('RE_CAMP_RUNTIME', '').strip().lower()
if not RUNTIME_NAME:
    RUNTIME_NAME = 'kaggle' if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle/working').is_dir() else 'colab'
CONTENT_ROOT = Path('/kaggle/working' if RUNTIME_NAME == 'kaggle' else '/content')
TOOLS_DIR = CONTENT_ROOT / 're-camp-blender'
ART_DIR = CONTENT_ROOT / 're-camp'
WONDER3D_DIR = CONTENT_ROOT / 'provider-wonder3D'
OUTPUT_ROOT = CONTENT_ROOT / 're-camp-ai3d' / CHARACTER_CODE / 'wonder3d'
REFERENCE_DIR = OUTPUT_ROOT / 'reference-views'
MULTIVIEW_DIR = OUTPUT_ROOT / 'multiview'
CANDIDATE_DIR = OUTPUT_ROOT / 'candidate'
EVALUATION_DIR = OUTPUT_ROOT / 'evaluation'
REVIEW_DIR = OUTPUT_ROOT / 'review'
assert CHARACTER_CODE == 'CH101'
print({'provider': 'wonder3D', 'commit': WONDER3D_COMMIT, 'inferenceScript': WONDER3D_INFERENCE_SCRIPT, 'generatedViewCount': WONDER3D_EXPECTED_VIEWS, 'unityInputAllowed': False})

In [ ]:
# Reuse a complete six-view output before requesting a new GPU allocation.
REUSE_WONDER3D = os.environ.get('RE_CAMP_REUSE_WONDER3D', '1') != '0'
MULTIVIEW_REPORT = MULTIVIEW_DIR / 'wonder3d-generation-report.json'
REUSE_MULTIVIEW = False
REUSE_CHECK = {'status': 'NOT_REUSABLE', 'reasons': ['REPORT_MISSING']}
def sha256_path(path):
    import hashlib
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
if REUSE_WONDER3D and MULTIVIEW_REPORT.is_file():
    existing = json.loads(MULTIVIEW_REPORT.read_text(encoding='utf-8'))
    reasons = []
    generation_status = existing.get('generationStatus', existing.get('status'))
    if generation_status != 'MULTIVIEW_GENERATED': reasons.append('GENERATION_STATUS_NOT_COMPLETE')
    if existing.get('provider') != 'wonder3D': reasons.append('PROVIDER_MISMATCH')
    if existing.get('providerCommit') != WONDER3D_COMMIT or existing.get('providerRepoHead') != WONDER3D_COMMIT: reasons.append('PROVIDER_COMMIT_MISMATCH')
    if existing.get('generatedViewCount') != WONDER3D_EXPECTED_VIEWS or existing.get('generatedAzimuths') != [0, 45, 90, 180, -90, -45]: reasons.append('VIEW_LAYOUT_MISMATCH')
    if existing.get('unityInputAllowed') is not False or existing.get('productionPromotionAllowed') is not False: reasons.append('GATE_UNLOCKED')
    existing_reference_manifest = REFERENCE_DIR / 'reference-views-manifest.json'
    if not existing_reference_manifest.is_file() or existing.get('referenceManifestSha256') != sha256_path(existing_reference_manifest): reasons.append('REFERENCE_MANIFEST_SHA256_MISMATCH')
    generated_files = existing.get('generatedFiles')
    if not isinstance(generated_files, list) or len(generated_files) < WONDER3D_EXPECTED_VIEWS: reasons.append('GENERATED_VIEW_FILES_INCOMPLETE')
    else:
        for relative_name in generated_files:
            generated_path = (MULTIVIEW_DIR / relative_name).resolve()
            try: generated_path.relative_to(MULTIVIEW_DIR.resolve())
            except ValueError: reasons.append('GENERATED_FILE_ESCAPES_OUTPUT'); continue
            if not generated_path.is_file(): reasons.append(f'GENERATED_FILE_MISSING:{relative_name}')
    REUSE_CHECK = {'status': 'REUSABLE' if not reasons else 'NOT_REUSABLE', 'reasons': reasons}
    REUSE_MULTIVIEW = not reasons
print({'reuseRequested': REUSE_WONDER3D, 'reuseMultiview': REUSE_MULTIVIEW, 'reuseCheck': REUSE_CHECK, 'unityInputAllowed': False})

In [ ]:
# Select GPU or No-GPU work before installing Blender/CUDA dependencies.
def bootstrap_run(command):
    print('RUN:', ' '.join(str(part) for part in command))
    return subprocess.run([str(part) for part in command], check=True)

bootstrap_run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow'])
if not (TOOLS_DIR / '.git').is_dir():
    bootstrap_run(['git', 'clone', '--branch', TOOLS_REF, TOOLS_REPO_URL, TOOLS_DIR])
bootstrap_run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_REF])
bootstrap_run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', f'origin/{TOOLS_REF}'])
if not (ART_DIR / '.git').is_dir():
    bootstrap_run(['git', 'clone', ART_REPO_URL, ART_DIR])
bootstrap_run(['git', '-C', ART_DIR, 'fetch', 'origin', ART_COMMIT])
bootstrap_run(['git', '-C', ART_DIR, 'checkout', '--detach', ART_COMMIT])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ADAPTIVE_REPORT_PATH = OUTPUT_ROOT / 'adaptive-workstream-report.json'
bootstrap_run([sys.executable, TOOLS_DIR / 'scripts' / 'run_adaptive_workstream.py', '--provider', 'wonder3D', '--art-root', ART_DIR, '--output', ADAPTIVE_REPORT_PATH])
adaptive_report = json.loads(ADAPTIVE_REPORT_PATH.read_text(encoding='utf-8'))
GPU_PREFLIGHT = adaptive_report['runtimePreflight']
GPU_COUNT = GPU_PREFLIGHT.get('gpuCount', 0)
GPU_PROBE = 'adaptive-runtime-preflight'
GPU_WORK_ENABLED = adaptive_report['selectedWorkstream'] == 'GPU'
print({'adaptiveStatus': adaptive_report['status'], 'selectedWorkstream': adaptive_report['selectedWorkstream'], 'reuseMultiview': REUSE_MULTIVIEW, 'gpuCount': GPU_COUNT, 'unityInputAllowed': False})
if not GPU_WORK_ENABLED:
    raise RuntimeError('ADAPTIVE_NO_GPU_COMPLETED: maintenance work finished; Wonder3D and NeuS remain blocked until GPU returns')


In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
def run(command, **kwargs):
    print('RUN:', ' '.join(str(part) for part in command))
    return subprocess.run([str(part) for part in command], check=True, **kwargs)

run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow'])
if shutil.which('blender') is None or shutil.which('xvfb-run') is None:
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', '-qq', 'blender', 'xvfb'])
for repo_url, repo_dir, ref in ((TOOLS_REPO_URL, TOOLS_DIR, TOOLS_REF), (ART_REPO_URL, ART_DIR, ART_COMMIT), (WONDER3D_REPO_URL, WONDER3D_DIR, WONDER3D_COMMIT)):
    if not (repo_dir / '.git').is_dir():
        run(['git', 'clone', repo_url, repo_dir])
    run(['git', '-C', repo_dir, 'fetch', 'origin', ref])
    run(['git', '-C', repo_dir, 'checkout', '--detach', ref])
assert subprocess.check_output(['git', '-C', WONDER3D_DIR, 'rev-parse', 'HEAD'], text=True).strip() == WONDER3D_COMMIT
if not REUSE_MULTIVIEW:
    wonder_requirements = []
    for requirement in (WONDER3D_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines():
        normalized_requirement = requirement.strip()
        if not normalized_requirement or normalized_requirement.startswith('#'):
            continue
        if normalized_requirement.lower().split('==', 1)[0].strip() in {'torch', 'torchvision', 'torchaudio'}:
            continue
        wonder_requirements.append(normalized_requirement)
    print({'skippedProviderTorchPins': True, 'remainingRequirementCount': len(wonder_requirements)})
    run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '--disable-pip-version-check', *wonder_requirements])
    run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch'])
else:
    print('REUSED_EXISTING_MULTIVIEW: skip Wonder3D and tiny-cuda-nn installation')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BLENDER_PYTHON_SITE = CONTENT_ROOT / 'blender-python-site'
BLENDER_PYTHON_SITE.mkdir(parents=True, exist_ok=True)
if not (BLENDER_PYTHON_SITE / 'numpy').is_dir():
    run([sys.executable, '-m', 'pip', 'install', '-q', '--target', BLENDER_PYTHON_SITE, '--platform', 'manylinux_2_17_x86_64', '--python-version', '3.10', '--only-binary=:all:', '--no-deps', 'numpy==1.23.5'])
BLENDER_ENVIRONMENT = os.environ.copy()
BLENDER_ENVIRONMENT['PYTHONPATH'] = str(BLENDER_PYTHON_SITE)

In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_reference_views.py', '--art-root', ART_DIR, '--output-dir', REFERENCE_DIR])
REFERENCE_MANIFEST = REFERENCE_DIR / 'reference-views-manifest.json'
reference_manifest = json.loads(REFERENCE_MANIFEST.read_text(encoding='utf-8'))
assert reference_manifest['artCommit'] == ART_COMMIT
assert reference_manifest['unityInputAllowed'] is False
FRONT_IMAGE = Path(reference_manifest['views']['front']['path'])
print({'reference': str(FRONT_IMAGE), 'referenceSha256': reference_manifest['views']['front']['sha256'], 'unityInputAllowed': False})

In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
reuse_args = ['--reuse-existing'] if REUSE_WONDER3D else []
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_wonder3d_multiview.py', '--provider-repo', WONDER3D_DIR, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', MULTIVIEW_DIR, '--execute'] + reuse_args)
MULTIVIEW_REPORT = MULTIVIEW_DIR / 'wonder3d-generation-report.json'
multiview_report = json.loads(MULTIVIEW_REPORT.read_text(encoding='utf-8'))
assert multiview_report['status'] in {'MULTIVIEW_GENERATED', 'REUSED'}
assert multiview_report['generationStatus'] == 'MULTIVIEW_GENERATED'
assert multiview_report['actualInference'] is False if multiview_report['status'] == 'REUSED' else True
assert multiview_report['unityInputAllowed'] is False
print(json.dumps(multiview_report, indent=2, ensure_ascii=False))

In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
# Wonder3D's official NeuS route is used because it is the lower-friction mesh extractor.
NEUS_DIR = WONDER3D_DIR / 'NeuS'
if not (NEUS_DIR / 'run.sh').is_file():
    raise FileNotFoundError(NEUS_DIR / 'run.sh')
run(['bash', 'run.sh', MULTIVIEW_DIR, 'CH101_front'], cwd=NEUS_DIR)
mesh_candidates = sorted(path for path in MULTIVIEW_DIR.rglob('*') if path.is_file() and path.suffix.lower() in {'.ply', '.obj', '.glb', '.gltf'})
if not mesh_candidates:
    raise RuntimeError('Wonder3D mesh extraction produced no supported mesh file')
MESH_PATH = mesh_candidates[0]
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_wonder3d_candidate.py', '--mesh', MESH_PATH, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', CANDIDATE_DIR])
CANDIDATE_MANIFEST = CANDIDATE_DIR / 'candidate-manifest.json'
candidate_manifest = json.loads(CANDIDATE_MANIFEST.read_text(encoding='utf-8'))
assert candidate_manifest['unityInputAllowed'] is False
print({'mesh': str(MESH_PATH), 'candidateManifest': str(CANDIDATE_MANIFEST), 'unityInputAllowed': False})

In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
entry = candidate_manifest['candidates'][0]
candidate_id = 'wonder3D-CH101-001'
candidate_output = EVALUATION_DIR / candidate_id
candidate_output.mkdir(parents=True, exist_ok=True)
launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
refined_glb = candidate_output / f'{candidate_id}_refined.glb'
refined_blend = candidate_output / f'{candidate_id}_refined_NOT_PRODUCTION.blend'
refinement_report = candidate_output / 'refinement-report.json'
run(launcher + ['blender', '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'refine_ai3d_candidate.py', '--', '--candidate', entry['modelPath'], '--output-glb', refined_glb, '--output-blend', refined_blend, '--report', refinement_report, '--provider', 'wonder3D', '--attempt', '1', '--parent-sha256', entry['sha256'], '--material-mode', 'preserve'], env=BLENDER_ENVIRONMENT)
evaluation_report = candidate_output / 'evaluation-report.json'
normalized_blend = candidate_output / f'{candidate_id}_normalized_NOT_PRODUCTION.blend'
run(launcher + ['blender', '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'evaluate_ai3d_candidate.py', '--', '--candidate', refined_glb, '--candidate-id', candidate_id, '--output-dir', candidate_output, '--report', evaluation_report, '--normalized-blend', normalized_blend, '--integrity-blend', refined_blend], env=BLENDER_ENVIRONMENT)
score_report = candidate_output / 'candidate-score.json'
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'score_candidate_renders.py', '--reference-manifest', REFERENCE_MANIFEST, '--evaluation-report', evaluation_report, '--output', score_report])
score_data = json.loads(score_report.read_text(encoding='utf-8'))
if score_data.get('orientationValidation', {}).get('correctionRequired'):
    print(f'Upside-down candidate detected; applying vertical polarity correction: {candidate_id}')
    run(launcher + ['blender', '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'refine_ai3d_candidate.py', '--', '--candidate', entry['modelPath'], '--output-glb', refined_glb, '--output-blend', refined_blend, '--report', refinement_report, '--provider', 'wonder3D', '--attempt', '1', '--parent-sha256', entry['sha256'], '--material-mode', 'preserve', '--invert-up-axis'], env=BLENDER_ENVIRONMENT)
    run(launcher + ['blender', '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'evaluate_ai3d_candidate.py', '--', '--candidate', refined_glb, '--candidate-id', candidate_id, '--output-dir', candidate_output, '--report', evaluation_report, '--normalized-blend', normalized_blend, '--integrity-blend', refined_blend], env=BLENDER_ENVIRONMENT)
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'score_candidate_renders.py', '--reference-manifest', REFERENCE_MANIFEST, '--evaluation-report', evaluation_report, '--output', score_report])
    score_data = json.loads(score_report.read_text(encoding='utf-8'))
    if score_data.get('orientationValidation', {}).get('correctionRequired'):
        raise RuntimeError(f'VERTICAL_POLARITY_CORRECTION_FAILED:{candidate_id}')
RANKING_MANIFEST = OUTPUT_ROOT / 'ranking-manifest.json'
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'rank_candidates.py', '--output', RANKING_MANIFEST, '--score-report', score_report])
ranking = json.loads(RANKING_MANIFEST.read_text(encoding='utf-8'))
assert ranking['unityInputAllowed'] is False
print(json.dumps(ranking, indent=2, ensure_ascii=False))

In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
archive_base = CONTENT_ROOT / f're-camp-{CHARACTER_CODE}-wonder3d-NOT-PRODUCTION'
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT))
print({'archive': str(archive_path), 'status': ranking.get('status'), 'selectedCandidate': ranking.get('selectedCandidate'), 'unityInputAllowed': ranking.get('unityInputAllowed')})
try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    print('Browser download is unavailable; copy the archive before the session ends.')